# Biohub Cell Tracking - SUBMISSION notebook (offline, no-internet safe)

Model-agnostic submission: runs a trained **3D U-Net detector** (heatmap regression) + Hungarian NN
linking over the test set, writes `/kaggle/working/submission.csv`. Point it at any weights version by
attaching that weights dataset -- the notebook is NOT tied to a specific model version.

**Before submitting:**
1. Add Input: the competition data, the `zarr3-offline` wheel dataset, and your **weights dataset**
   (a `.pt` such as `v1_UNet_best.pt`). The model architecture below MUST match the one that produced
   the weights (base channels / strides).
2. Turn **Internet OFF** and Run All to VERIFY the offline zarr install + weight load work end-to-end.
3. Save Version, then Submit to Competition.

**Density-penalty knob:** `HM_THR` controls detection count. Lower = more detections = higher base
Jaccard locally, but the leaderboard's *adjusted* score penalises over-prediction
(`x (1 - 0.1*(T_pred - T_true)/T_true)`). If the LB score is dragged below the local edge-Jaccard,
raise `HM_THR`. Calibrate via the leaderboard.


In [ ]:
# Offline zarr>=3 install: auto-find the attached wheel dataset (no internet needed).
import glob, os, subprocess, sys
def find_wheeldir():
    for whl in glob.glob('/kaggle/input/**/zarr*.whl', recursive=True):
        return os.path.dirname(whl)
    return None
try:
    import zarr
    assert int(zarr.__version__.split('.')[0]) >= 3
    print('zarr already present:', zarr.__version__)
except Exception:
    wd = find_wheeldir()
    assert wd, 'No zarr wheel found under /kaggle/input -- attach the zarr3-offline dataset!'
    print('installing zarr offline from', wd)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', wd, 'zarr'])
    import zarr
    print('installed zarr:', zarr.__version__)


In [ ]:
import numpy as np, pandas as pd, time
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max

# ---------------- CONFIG ----------------
VOXEL   = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um per voxel (anisotropic)

# detection / linking (match the values the model was evaluated at)
NORM_PCT     = (50.0, 99.8)   # percentile normalisation, must match training
HM_THR       = 0.3            # heatmap peak threshold -> density knob (see markdown; raise if LB penalised)
MIN_DISTANCE = 3              # peak_local_max separation (voxels)
MAX_PEAKS    = 5000           # per-frame cap
GATE_UM      = 8.0            # NN linking gate (um)

# model architecture -- MUST match the training notebook that produced the weights
BASE    = 16
STRIDES = ((1, 2, 2), (1, 2, 2), (2, 2, 2), (2, 2, 2))   # downsample xy first, z later

# weights: None = auto-find under /kaggle/input (v1_UNet_best.pt / unet_heatmap.pt / unet_latest.pt / *.pt)
WEIGHTS_PATH = None

INPUT    = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = os.path.join(INPUT, 'test')
OUT      = '/kaggle/working/submission.csv'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)


In [ ]:
# ---------------- IO + model + inference (mirrors src/v1_unet_train.ipynb) ----------------
def open_image(zarr_path):
    """Return the (T,Z,Y,X) array of an OME-Zarr sample (largest array)."""
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g)
    return best

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)   # raw logits; sigmoid at inference

def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)   # keep float32 (AMP-safe)

@torch.no_grad()
def predict_heatmap(model, vol, norm_pct=NORM_PCT):
    v = _normalize(vol, norm_pct)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def detect_all_unet(model, arr, threshold=HM_THR, min_distance=MIN_DISTANCE, max_peaks=MAX_PEAKS):
    model.eval(); out = []
    for t in range(arr.shape[0]):
        hm = predict_heatmap(model, np.asarray(arr[t]))
        out.append(peak_local_max(hm, min_distance=min_distance, threshold_abs=threshold,
                                  exclude_border=False, num_peaks=max_peaks))
    return out

def build_graph(coords_list, gate=GATE_UM):
    nodes, edges, nid = [], [], 0
    prev_ids, prev_xyz = None, None
    for t, coords in enumerate(coords_list):
        ids = list(range(nid, nid+len(coords))); nid += len(coords)
        for i, c in zip(ids, coords): nodes.append((i, t, int(c[0]), int(c[1]), int(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz*VOXEL, coords*VOXEL)
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= gate: edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges


In [ ]:
# ---------------- Load weights (auto-find; handles state_dict OR full checkpoint) ----------------
def find_weights():
    if WEIGHTS_PATH:
        return WEIGHTS_PATH
    for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits:
            return hits[0]
    return None

def extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

wpath = find_weights()
assert wpath, 'No .pt weights found under /kaggle/input -- attach your weights dataset (e.g. v1_UNet_best.pt).'
model = UNet3D().to(device)
model.load_state_dict(extract_state(torch.load(wpath, map_location=device)))
model.eval()
print('loaded weights:', wpath)


In [ ]:
# ---------------- Generate submission for ALL test datasets ----------------
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('test datasets:', len(test_zarrs))

rows = []; gid = 0; t0 = time.time()
for zp in test_zarrs:
    ds = os.path.basename(zp)[:-5]
    arr = open_image(zp)
    coords = detect_all_unet(model, arr)
    nodes, edges = build_graph(coords, gate=GATE_UM)
    for (nid, t, z, y, x) in nodes:
        rows.append((gid, ds, 'node', nid, t, z, y, x, -1, -1)); gid += 1
    for (s, tg) in edges:
        rows.append((gid, ds, 'edge', -1, -1, -1, -1, -1, s, tg)); gid += 1
    print(f'  {ds}: {len(nodes)} nodes, {len(edges)} edges  ({time.time()-t0:.0f}s)')

cols = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
sub = pd.DataFrame(rows, columns=cols)
sub.to_csv(OUT, index=False)
print('\nwrote', OUT, sub.shape)
print(sub.head(6).to_string(index=False))
